# Curation Analysis

Visualise what the curation pipeline accepts/rejects and why.

In [ ]:
import sys
sys.path.insert(0, '../src')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid')

MANIFEST = Path('../data/curated/manifest.jsonl')

if MANIFEST.exists():
    with open(MANIFEST) as f:
        clips = [json.loads(l) for l in f]
    df = pd.DataFrame(clips)
    print(f'Loaded {len(df)} curated clips')
    print(df.describe()[['blur_score','motion_score','quality_score']])
else:
    print(f'Manifest not found: {MANIFEST}')
    print('Run scripts/run_curation.py first')

In [ ]:
# Score distributions by class
if 'df' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    for ax, col, label in zip(axes,
        ['blur_score', 'motion_score', 'quality_score'],
        ['Blur Score (higher = sharper)', 'Motion Score', 'BRISQUE Quality (lower = better)']):
        sns.violinplot(data=df, x='label', y=col, ax=ax, inner='box')
        ax.set_title(label)
        ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.savefig('../results/bias_analysis/curation_score_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# Blur score vs motion score scatter — identify at-risk classes
if 'df' in dir():
    plt.figure(figsize=(9, 7))
    at_risk = ['PlayingGuitar', 'Rowing']
    colors = {lbl: 'crimson' if lbl in at_risk else 'steelblue' for lbl in df['label'].unique()}

    for lbl, grp in df.groupby('label'):
        plt.scatter(grp['blur_score'], grp['motion_score'],
                    label=lbl, c=colors[lbl],
                    alpha=0.7 if lbl in at_risk else 0.3,
                    s=30 if lbl not in at_risk else 60,
                    zorder=3 if lbl in at_risk else 1)

    plt.axvline(x=80, linestyle='--', color='purple', alpha=0.6, label='σ=80 threshold')
    plt.axvline(x=40, linestyle='--', color='gray', alpha=0.5, label='σ=40 threshold')
    plt.xlabel('Blur Score (Laplacian σ)')
    plt.ylabel('Motion Score (mean flow magnitude)')
    plt.title('Clip Quality Space\n(red = at-risk classes for blur filtering)')
    plt.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()